# MoveScope 视角鲁棒性实验

检查完整三维评估在正面、侧面和斜向深蹲视频上是否比二维基线更稳定。

In [ ]:
from pathlib import Path
import sys

ROOT = next(path for path in [Path.cwd(), *Path.cwd().parents] if (path / "movescope").exists())
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import matplotlib.pyplot as plt
import pandas as pd
from scipy import stats

from movescope.experiments import run_ablation, summarize_ablation, run_viewpoint_consistency, viewpoint_std
from movescope.template import ActionTemplate

FIG_DIR = ROOT / "docs" / "figures"
FIG_DIR.mkdir(parents=True, exist_ok=True)


In [ ]:
template_path = Path("data/templates/squat.npz")
if not template_path.exists():
    print("缺少 data/templates/squat.npz，请先运行 scripts/build_template.py 构建模板。")
    rows = []
else:
    template = ActionTemplate.load("squat")
    rows = run_viewpoint_consistency(template, multiview_dir="data/test/multiview")
    print(f"已收集 {len(rows)} 条评分记录")


In [ ]:
df = pd.DataFrame(rows)
if df.empty:
    print("未找到多视角视频，请先向 data/test/multiview 添加正面、侧面和斜向视频。")
else:
    display(df)
    print(viewpoint_std(rows))


In [ ]:
if not df.empty:
    fig, ax = plt.subplots(figsize=(8, 4))
    for variant, group in df.groupby("variant"):
        group = group.sort_values("angle")
        ax.plot(group["angle"], group["total_score"], marker="o", label=variant)
    ax.set_title("不同相机视角下的评分一致性")
    ax.set_xlabel("拍摄视角")
    ax.set_ylabel("总分")
    ax.legend()
    fig.tight_layout()
    output = FIG_DIR / "viewpoint_robustness.png"
    fig.savefig(output, dpi=160)
    print(f"图表已保存：{output}")


In [ ]:
if not df.empty:
    stds = viewpoint_std(rows)
    baseline = stds.get("baseline_2d")
    full = stds.get("ours_full")
    if baseline and full is not None:
        improvement = (baseline - full) / baseline * 100.0
        print(f"ours_full 的评分一致性提升：{improvement:.1f}%")
